# Lumora Chat Explorer
Explore message balance, conversation intensity, reply rhythm, recurring topics, and tone signals from anonymized WhatsApp exports.

**Steps:** run all cells (Runtime → Run all) and upload `sample_chat_messages.csv` when asked. Each chart has its own **Chat** dropdown, so you can compare one conversation with all of them.

To create the CSV from local ZIP exports, run `python colab/prepare_chat_samples.py` in the Lumora project, then upload `colab/sample_chat_messages.csv`. The CSV is excluded from Git because it contains private message text.

> **Privacy:** This notebook analyzes message text in your Colab runtime. Upload only chats you have permission to analyze. The prepared CSV pseudonymizes participants (including stretched spellings of their names) and scrubs common phone, email, and URL patterns, but it may still contain identifying details. Review it before sharing, and do not publish notebook outputs that show message text.

Emotion and topic charts are keyword estimates. They do not establish anyone's actual feelings or intent.

In [ ]:
import io
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Markdown

PALETTE = ['#7556E8', '#F39B79', '#77BFA3', '#78A9E8']
BG = '#FBF9FF'
sns.set_theme(style='whitegrid', palette=PALETTE)

try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(name for name in uploaded if name.lower().endswith('.csv'))
    df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
except ImportError:
    csv_name = 'sample_chat_messages.csv'
    df = pd.read_csv(csv_name)

required = {'conversation_id', 'timestamp', 'sender', 'text'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required CSV columns: {sorted(missing)}')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['text'] = df['text'].fillna('').astype(str)
df = df.dropna(subset=['timestamp']).sort_values(['conversation_id', 'timestamp']).reset_index(drop=True)
if 'is_media' not in df.columns:  # older CSVs: detect stickers/photos/deleted messages from the placeholder text
    df['is_media'] = df['text'].str.contains(r'omitted|tidak disertakan|dihapus|was deleted', case=False, regex=True)
df['is_media'] = df['is_media'].astype(bool)
# participant_1 in one chat is a different person from participant_1 in another chat.
df['speaker'] = df['conversation_id'].str.replace('conversation_', 'c', regex=False) + ' · ' + df['sender']
df['person'] = df['sender'].str.replace('participant_', 'Person ', regex=False)
df['message_length'] = df['text'].str.len().where(~df['is_media'])
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
print(f'Loaded {len(df):,} messages from {df.conversation_id.nunique():,} conversations '
      f'({df.speaker.nunique():,} participants, {int(df.is_media.sum()):,} media/deleted placeholders).')

## Conversation overview
One row per chat. Participants are labelled per chat: Person 1 in one chat is not the same person as Person 1 in another.

In [ ]:
CHAT_OPTIONS = ['All conversations'] + sorted(df['conversation_id'].unique().tolist())

def rows_for(chat):
    return df if chat == 'All conversations' else df[df['conversation_id'] == chat]

def who(data):
    # Inside one chat, show Person 1 / Person 2. Across chats, keep people separate.
    return data['person'] if data['conversation_id'].nunique() == 1 else data['speaker']

def chat_picker():
    return widgets.Dropdown(options=CHAT_OPTIONS, value='All conversations', description='Chat:',
                            layout=widgets.Layout(width='320px'))

overview = (df.groupby('conversation_id')
              .agg(messages=('text', 'size'), media=('is_media', 'sum'),
                   first=('timestamp', 'min'), last=('timestamp', 'max'))
              .assign(days=lambda t: (t['last'] - t['first']).dt.days + 1,
                      per_day=lambda t: (t['messages'] / t['days']).round(1)))
overview['first'] = overview['first'].dt.strftime('%d %b %Y')
overview['last'] = overview['last'].dt.strftime('%d %b %Y')
display(overview.rename(columns={'per_day': 'messages/day'}))
# Structural fields only; message text stays out of notebook output.
display(df[['conversation_id', 'timestamp', 'sender', 'is_media', 'message_length']].head(8))

## Message balance
Who sends more messages, and who writes longer ones.

In [ ]:
@widgets.interact(chat=chat_picker())
def draw_balance(chat):
    data = rows_for(chat)
    names = who(data)
    counts = names.value_counts()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=BG)
    if len(counts) <= 6:
        ax1.pie(counts.values, labels=counts.index, autopct='%1.0f%%', startangle=90,
                colors=sns.color_palette(PALETTE, len(counts)),
                wedgeprops={'width': .42, 'edgecolor': 'white', 'linewidth': 2})
    else:
        counts.head(12).sort_values().plot.barh(ax=ax1, color=PALETTE[0])
        ax1.set_xlabel('Messages')
    ax1.set_title('Messages by participant', weight='bold')
    avg = data.assign(who=names).groupby('who')['message_length'].mean().reindex(counts.index).head(12)
    colors = dict(zip(counts.index, sns.color_palette(PALETTE, len(counts)))) if len(counts) <= 6 else {}
    avg = avg.sort_values()
    avg.plot.barh(ax=ax2, color=[colors.get(name, PALETTE[1]) for name in avg.index])
    ax2.set(title='Average characters per text message', xlabel='Characters', ylabel='')
    ax2.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

## Conversation intensity

In [ ]:
@widgets.interact(chat=chat_picker(), period=widgets.ToggleButtons(options=[('Weekly', 'W'), ('Monthly', 'M')], description='Group by:'))
def draw_intensity(chat, period):
    data = rows_for(chat)
    rule = 'W-MON' if period == 'W' else 'MS'
    # Resample so quiet weeks/months count as 0 instead of being bridged by a line.
    if chat == 'All conversations':
        series = {'All chats': data.set_index('timestamp').resample(rule).size()}
    else:
        series = {name: group.set_index('timestamp').resample(rule).size()
                  for name, group in data.groupby(who(data))}
    fig, ax = plt.subplots(figsize=(12, 4.5), facecolor=BG)
    for color, (name, values) in zip(PALETTE, series.items()):
        ax.plot(values.index, values.values, marker='o', markersize=3.5, linewidth=2.2, color=color, label=name)
        ax.fill_between(values.index, values.values, alpha=.12, color=color)
    ax.set(title='Messages over time', xlabel='', ylabel='Messages')
    ax.legend(frameon=False, ncol=len(series))
    ax.spines[['top', 'right']].set_visible(False)
    plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

## Reply rhythm
Median time before someone answers the other person. Gaps over 6 hours are treated as a new conversation, not a slow reply.

In [ ]:
def reply_times(data):
    rows = []
    for _, chat in data.groupby('conversation_id'):
        chat = chat.sort_values('timestamp')
        gap = chat['timestamp'].diff().dt.total_seconds() / 60
        switched = chat['sender'] != chat['sender'].shift()
        mask = switched & gap.notna() & (gap <= 360)
        rows.append(pd.DataFrame({'who': who(chat)[mask], 'minutes': gap[mask]}))
    return pd.concat(rows) if rows else pd.DataFrame(columns=['who', 'minutes'])

@widgets.interact(chat=chat_picker())
def draw_replies(chat):
    replies = reply_times(rows_for(chat))
    if replies.empty:
        print('Not enough replies to measure.'); return
    median = replies.groupby('who')['minutes'].median().sort_values().head(15)
    fig, ax = plt.subplots(figsize=(10, max(2.5, .45 * len(median) + 1)), facecolor=BG)
    median.plot.barh(ax=ax, color=PALETTE[2])
    for i, value in enumerate(median.values):
        ax.text(value, i, f' {value:.1f} min', va='center', fontsize=9)
    ax.set(title='Median reply time', xlabel='Minutes', ylabel='')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

## Topic signals
A message can match more than one topic. Counts are whole-word keyword matches, not semantic classification. Media placeholders are ignored.

In [ ]:
TOPICS = {
    'Plans & logistics': ['tomorrow', 'today', 'meet', 'where', 'when', 'besok', 'nanti', 'ketemu', 'jadwal', 'dimana'],
    'Campus & classes': ['class', 'lecture', 'course', 'campus', 'kampus', 'dosen', 'kuliah', 'kelas', 'matkul', 'skripsi', 'semester'],
    'Work & projects': ['work', 'project', 'meeting', 'proposal', 'lpj', 'hima', 'event', 'kerja', 'proyek', 'acara'],
    'Support & advice': ['advice', 'help', 'worry', 'stress', 'capek', 'cerita', 'saran', 'bantu', 'bingung', 'semangat'],
    'Food & everyday life': ['eat', 'food', 'lunch', 'dinner', 'makan', 'minum', 'resto', 'kopi', 'enak'],
}

def word_pattern(terms):
    return r'\b(?:' + '|'.join(re.escape(term) for term in terms) + r')\b'

@widgets.interact(chat=chat_picker())
def draw_topics(chat):
    text = rows_for(chat).loc[lambda d: ~d['is_media'], 'text']
    counts = pd.Series({topic: int(text.str.contains(word_pattern(terms), case=False, regex=True).sum())
                        for topic, terms in TOPICS.items()}).sort_values()
    fig, ax = plt.subplots(figsize=(9, 4), facecolor=BG)
    counts.plot.barh(ax=ax, color=sns.color_palette('pastel', len(counts)))
    for i, value in enumerate(counts.values):
        ax.text(value, i, f' {value:,} ({value / max(len(text), 1):.0%})', va='center', fontsize=9)
    ax.set(title='Recurring topic keyword signals', xlabel='Messages containing a keyword', ylabel='')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

## Tone keyword signals
These count visible wording and laughter only. They do not establish a participant's real emotion or intent.

In [ ]:
TONE = {
    'Excited': ['excited', 'happy', 'seneng', 'senang', 'seru', 'asik', 'yay', 'mantap'],
    'Angry': ['angry', 'marah', 'kesel', 'kesal', 'sebel', 'annoyed', 'bete'],
    'Sad': ['sad', 'sedih', 'nangis', 'cry', 'crying', 'kecewa', 'galau'],
}
LAUGH = r'\b(?:w+k+w+[wk]*|(?:ha){2,}h?|(?:he){2,}h?|(?:hi){2,}h?|lol)\b|😂|🤣'

@widgets.interact(chat=chat_picker())
def draw_tone(chat):
    data = rows_for(chat).loc[lambda d: ~d['is_media']].copy()
    data['who'] = who(data)
    top = data['who'].value_counts().head(8).index
    rows = []
    for name, group in data[data['who'].isin(top)].groupby('who'):
        text = group['text'].str.lower()
        for label, terms in TONE.items():
            rows.append({'Participant': name, 'Signal': label,
                         'Messages (%)': text.str.contains(word_pattern(terms), regex=True).mean() * 100})
        rows.append({'Participant': name, 'Signal': 'Laughter',
                     'Messages (%)': text.str.contains(LAUGH, regex=True).mean() * 100})
    tone = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(11, 4.5), facecolor=BG)
    sns.barplot(data=tone, x='Participant', y='Messages (%)', hue='Signal',
                palette=['#F0A078', '#8B69E8', '#83AEE8', '#77BFA3'], ax=ax)
    ax.set(title='Messages with tone keywords', ylabel='Share of text messages (%)', xlabel='')
    ax.spines[['top', 'right']].set_visible(False); ax.legend(frameon=False)
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## When conversations happen

In [ ]:
@widgets.interact(chat=chat_picker())
def draw_heatmap(chat):
    data = rows_for(chat)
    order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    grid = (data.pivot_table(index='weekday', columns='hour', values='text', aggfunc='count', fill_value=0)
                .reindex(index=order, columns=range(24)).fillna(0))
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG)
    sns.heatmap(grid, cmap='Purples', linewidths=.4, ax=ax, cbar_kws={'label': 'Messages'})
    ax.set(title='Message activity by weekday and hour', xlabel='Hour of day', ylabel='')
    plt.tight_layout(); plt.show()

## Reading the charts
Use the charts as descriptive summaries. Message volume, keyword matches, and reply timing can show patterns in the export, but they cannot prove relationship quality, intent, or private feelings. Remove identifying details before sharing notebook outputs.